# OpenDecider on a free Colab GPU

Open, calibrated System 1 decision models: ask typed questions (`choice`, `score`, `noul` = yes/no) about any text or
JSON and get a calibrated probability for every option.

* **opendecider-nano** (~400M): 16 ms per question on an NVIDIA GPU; beats Laya and TypeSafe Jev on typed-decisions (0.796 vs 0.766 and 0.754).
* **opendecider-small** (4B): the most accurate on decisions it has never seen; fits a T4.

[GitHub](https://github.com/manjunathshiva/opendecider) · [Models](https://huggingface.co/collections/manjunathshiva/opendecider-6ab8c838909092518d50a9ea) · [Live demo](https://huggingface.co/spaces/manjunathshiva/opendecider-demo)

**Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

In [ ]:
!pip install -q "opendecider[small]"
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU: Runtime > Change runtime type > T4 GPU")

## 1. opendecider-nano: three typed questions about a support ticket

In [ ]:
from opendecider import load, Choice, Score, Noul

nano = load("manjunathshiva/opendecider-nano")

state = "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."
questions = {
    "department": Choice("Which department should handle this?",
                         {"billing": "invoices, payments, refunds", "technical": "bugs, outages, system errors",
                          "other": "everything else"}),
    "urgency": Score("How urgent is this?", ["not urgent", "soon", "blocking"]),
    "churn_risk": Noul("Does the user threaten to cancel or leave?"),
}
r = nano.system_one(state, questions)
for name, a in r["answers"].items():
    print(f"{name:11s} {a['type']:6s}", a.get("choice", a.get("score", a.get("noul"))), a["probabilities"])
print(r["latency_ms"], "ms for", len(questions), "questions")

## 2. Any JSON state: an AI agent's trace

In [ ]:
trace = {"task": "Book a table for 2 at an Italian restaurant tomorrow 7pm",
         "steps": [{"tool": "search_restaurants", "result": "3 results"},
                   {"tool": "book_table", "args": {"restaurant": "Trattoria Roma", "time": "19:00"}, "result": "error: card declined"},
                   {"tool": "book_table", "args": {"restaurant": "Trattoria Roma", "time": "19:00"}, "result": "error: card declined"}]}
r = nano.system_one(trace, {
    "next_action": Choice("What should the orchestrator do next?",
                          {"continue": "let the agent keep going", "retry": "retry the last step",
                           "ask_user": "stop and ask the user for help", "abort": "stop the task"}),
    "looping": Noul("Is the agent stuck repeating a failing step?"),
})
for name, a in r["answers"].items():
    print(name, a["probabilities"])

## 3. Speed on this GPU: 1, 5, 10 and 50 questions per call

In [ ]:
!python -m opendecider.bench_speed manjunathshiva/opendecider-nano

## 4. opendecider-small (4B)

Downloads Qwen3-4B-Instruct-2507 (8 GB) plus the adapter; about 2 minutes on Colab. On a T4 it runs in fp16
(the T4 has no bf16), which moves probabilities by about 0.003 against bf16.

In [ ]:
del nano; torch.cuda.empty_cache()
small = load("manjunathshiva/opendecider-small")
r = small.system_one({"message": "I took out cash abroad and the exchange rate is wrong."},
                     {"intent": Choice("Which banking intent is this?",
                                       ["wrong_exchange_rate_for_cash_withdrawal", "card_payment_fee_charged",
                                        "cash_withdrawal_charge", "declined_cash_withdrawal"]),
                      "complaint": Noul("Is the customer complaining?")})
for name, a in r["answers"].items():
    print(name, a["probabilities"])
print(f"GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")

## 5. Reproduce the benchmark tables

Rebuilds every table in the README from the committed results (no model runs), with the same metric code.
To re-score a model yourself: `python benchmarks/run.py --model opendecider-nano --suites typed`.

In [ ]:
!git clone -q https://github.com/manjunathshiva/opendecider.git
!pip install -q -r opendecider/benchmarks/requirements.txt
!python opendecider/benchmarks/report.py